# Participant name matching

This notebook compares participant names extracted from `outputs/**/*.json` against the per-conference truth files in `analysis_v1/data/*/*_outcome.json`, then writes the matched and unmatched participant CSVs into `analysis_v1/`. Name corrections are optional and only applied if a mapping file exists.

In [ ]:
import json
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

def resolve_base():
    cwd = Path.cwd().resolve()

    if cwd.name == "analysis_v1":
        return cwd.parent

    if (cwd / "analysis_v1").exists() and (cwd / "outputs").exists():
        return cwd

    if (cwd.parent / "analysis_v1").exists() and (cwd.parent / "outputs").exists():
        return cwd.parent

    raise FileNotFoundError(
        f"Could not locate the repo root from current working directory: {cwd}. "
        "Run the notebook from the repo root or from analysis_v1/ under it."
    )

BASE = resolve_base()
REPO_ROOT = BASE
OUTPUTS_ROOT = REPO_ROOT / "outputs"
TRUTH_ROOT = REPO_ROOT / "analysis_v1" / "data"
NAME_MAPPING_CANDIDATES = [
    REPO_ROOT / "analysis_v1" / "name_corrections_mapping.json", 
    REPO_ROOT / "name_corrections_mapping.json",
]
MANUAL_ALIAS_CANDIDATES = [
    REPO_ROOT / "analysis_v1" / "participant_alias_mapping.csv",
    REPO_ROOT / "participant_alias_mapping.csv",
]

if not OUTPUTS_ROOT.exists():
    raise FileNotFoundError(f"Outputs folder not found: {OUTPUTS_ROOT}")

if not TRUTH_ROOT.exists():
    raise FileNotFoundError(f"Truth data folder not found: {TRUTH_ROOT}")

def load_json(path: Path):
    try:
        with path.open("r", encoding="utf-8") as handle:
            return json.load(handle)
    except json.JSONDecodeError as exc:
        print(f"Skipping invalid JSON file: {path} ({exc})")
        return None

def strip_unicode(value):
    text = unicodedata.normalize("NFKD", value)
    return text.encode("ascii", "ignore").decode("ascii")

def normalize_name(value):
    if not isinstance(value, str):
        return ""
    text = strip_unicode(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

NAME_PREFIXES = {
    "dr",
    "prof",
    "professor",
    "mr",
    "mrs",
    "ms",
    "miss",
    "mx",
}

NAME_SUFFIXES = {
    "phd",
    "md",
    "m d",
    "dds",
    "dmd",
    "do",
    "msc",
    "ma",
    "ba",
    "bs",
    "mba",
    "jd",
    "esq",
}

AFFILIATION_WORDS = {
    "university",
    "univ",
    "college",
    "institute",
    "school",
    "department",
    "dept",
    "center",
    "centre",
    "laboratory",
    "lab",
    "labs",
    "hospital",
    "company",
    "industry",
    "research",
    "medicine",
    "med",
    "engineering",
}

ROLE_WORDS = {
    "she",
    "her",
    "he",
    "him",
    "they",
    "them",
    "their",
    "his",
    "hers",
    "speaker",
    "moderator",
    "participant",
}

def clean_name_for_matching(value):
    if not isinstance(value, str):
        return ""

    text = strip_unicode(value).lower().strip()
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\b(?:dr|prof|professor|mr|mrs|ms|miss|mx)\.?\s+", "", text)
    text = re.sub(r"[|;/]+", " ", text)
    text = re.sub(r"\s*,\s*", ",", text)
    text = re.sub(r"\s+", " ", text).strip()

    parts = [part.strip() for part in text.split(",") if part.strip()]
    if parts:
        text = parts[0]
    else:
        text = text.replace(",", " ")

    tokens = text.split()
    if not tokens:
        return ""

    while tokens and tokens[0].rstrip(".") in NAME_PREFIXES:
        tokens.pop(0)

    while tokens and tokens[-1].rstrip(".") in NAME_SUFFIXES.union(ROLE_WORDS):
        tokens.pop()

    cutoff = len(tokens)
    for index, token in enumerate(tokens):
        if token.rstrip(".") in AFFILIATION_WORDS:
            cutoff = index
            break

    if cutoff > 0:
        tokens = tokens[:cutoff]

    while tokens and tokens[-1].rstrip(".") in NAME_SUFFIXES.union(ROLE_WORDS):
        tokens.pop()

    return normalize_name(" ".join(tokens))

def build_corrections(mapping_data):
    corrections = {}

    def add(alias, canonical):
        alias_norm = clean_name_for_matching(alias)
        canonical_norm = clean_name_for_matching(canonical)
        if alias_norm and canonical_norm:
            corrections[alias_norm] = canonical_norm

    if isinstance(mapping_data, dict):
        for key, value in mapping_data.items():
            if isinstance(value, str):
                add(key, value)
            elif isinstance(value, list):
                for alias in value:
                    add(alias, key)
            elif isinstance(value, dict):
                canonical = (
                    value.get("corrected_name")
                    or value.get("canonical_name")
                    or value.get("canonical")
                    or value.get("name")
                    or key
                )
                aliases = (
                    value.get("aliases")
                    or value.get("variants")
                    or value.get("alternate_names")
                    or value.get("original_names")
                    or value.get("from")
                    or value.get("alias")
                )
                if isinstance(aliases, list):
                    for alias in aliases:
                        add(alias, canonical)
                else:
                    add(key, canonical)
    elif isinstance(mapping_data, list):
        for item in mapping_data:
            if not isinstance(item, dict):
                continue
            alias = item.get("alias") or item.get("from") or item.get("wrong") or item.get("name")
            canonical = item.get("corrected_name") or item.get("correct") or item.get("to") or item.get("canonical_name")
            add(alias, canonical)

    return corrections

def load_manual_alias_mapping():
    alias_path = next((path for path in MANUAL_ALIAS_CANDIDATES if path.exists()), None)
    if alias_path is None:
        return {}, None

    alias_df = pd.read_csv(alias_path)
    if alias_df.empty:
        return {}, alias_path

    alias_columns = {column.lower(): column for column in alias_df.columns}
    if "alias_name" not in alias_columns or "canonical_name" not in alias_columns:
        raise ValueError(
            f"Manual alias file {alias_path} must contain alias_name and canonical_name columns."
        )

    alias_name_column = alias_columns["alias_name"]
    canonical_name_column = alias_columns["canonical_name"]
    manual_aliases = {}
    for _, row in alias_df.iterrows():
        alias_name = row.get(alias_name_column)
        canonical_name = row.get(canonical_name_column)
        if pd.isna(alias_name) or pd.isna(canonical_name):
            continue
        alias_clean = clean_name_for_matching(str(alias_name))
        canonical_clean = clean_name_for_matching(str(canonical_name))
        if alias_clean and canonical_clean:
            manual_aliases[alias_clean] = canonical_clean

    return manual_aliases, alias_path

mapping_path = next((path for path in NAME_MAPPING_CANDIDATES if path.exists()), None)
if mapping_path is None:
    NAME_CORRECTIONS = {}
else:
    mapping_data = load_json(mapping_path)
    NAME_CORRECTIONS = build_corrections(mapping_data) if mapping_data is not None else {}

MANUAL_ALIAS_CORRECTIONS, MANUAL_ALIAS_PATH = load_manual_alias_mapping()

def canonicalize_name(value):
    normalized = clean_name_for_matching(value)
    if not normalized:
        return ""
    normalized = NAME_CORRECTIONS.get(normalized, normalized)
    normalized = MANUAL_ALIAS_CORRECTIONS.get(normalized, normalized)
    return normalized

In [19]:
TARGET_NAME_FIELDS = {
    "name",
    "participant",
    "participants",
    "participant_name",
    "full_name",
    "speaker",
    "speakers",
    "member",
    "members",
    "founder",
    "founders",
    "founder_name",
    "person",
    "people",
}

def collect_output_name_rows(data, conference, source_file):
    rows = []

    chunk_summary = data.get("chunk_summary") or {}
    speaking_time_seconds = chunk_summary.get("speaking_time_seconds") or {}
    if isinstance(speaking_time_seconds, dict):
        for name in speaking_time_seconds.keys():
            rows.append((conference, name, "chunk_summary.speaking_time_seconds", source_file))

    utterance_annotations = data.get("utterance_annotations") or []
    if isinstance(utterance_annotations, list):
        for index, utterance in enumerate(utterance_annotations):
            if isinstance(utterance, dict):
                speaker = utterance.get("speaker")
                if isinstance(speaker, str):
                    rows.append((conference, speaker, f"utterance_annotations[{index}].speaker", source_file))

    session_state = data.get("session_state") or {}
    speakers_identified = session_state.get("speakers_identified") or []
    if isinstance(speakers_identified, list):
        for speaker in speakers_identified:
            if isinstance(speaker, str):
                rows.append((conference, speaker, "session_state.speakers_identified", source_file))

    return rows

def extract_truth_name_rows(obj, conference, source_path="root"):
    rows = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            key_name = str(key).strip().lower()
            if key_name in TARGET_NAME_FIELDS:
                rows.extend(extract_names_from_value(value, conference, f"{source_path}.{key}"))
            rows.extend(extract_truth_name_rows(value, conference, f"{source_path}.{key}"))
    elif isinstance(obj, list):
        for index, item in enumerate(obj):
            rows.extend(extract_truth_name_rows(item, conference, f"{source_path}[{index}]"))

    return rows

def extract_names_from_value(value, conference, source_path):
    rows = []

    if isinstance(value, str):
        rows.append((conference, value, source_path))
    elif isinstance(value, list):
        for index, item in enumerate(value):
            if isinstance(item, str):
                rows.append((conference, item, source_path))
            elif isinstance(item, (dict, list)):
                rows.extend(extract_truth_name_rows(item, conference, f"{source_path}[{index}]"))
    elif isinstance(value, dict):
        rows.extend(extract_truth_name_rows(value, conference, source_path))

    return rows

def register_occurrence(store, conference, raw_name, source_label, source_file):
    canonical_name = canonicalize_name(raw_name)
    if not canonical_name:
        return

    key = (conference, canonical_name)
    entry = store[key]
    entry["raw_names"].add(raw_name.strip())
    entry["source_labels"].add(source_label)
    entry["source_files"].add(str(source_file.relative_to(REPO_ROOT)))
    entry["occurrence_count"] += 1

def make_empty_entry():
    return {
        "raw_names": set(),
        "source_labels": set(),
        "source_files": set(),
        "occurrence_count": 0,
    }

def store_to_dataframe(store, prefix):
    columns = [
        "conference",
        "normalized_name",
        f"{prefix}_raw_names",
        f"{prefix}_sources",
        f"{prefix}_source_files",
        f"{prefix}_occurrence_count",
        f"{prefix}_unique_file_count",
    ]
    records = []
    for (conference, normalized_name), payload in sorted(store.items()):
        records.append({
            "conference": conference,
            "normalized_name": normalized_name,
            f"{prefix}_raw_names": " | ".join(sorted(payload["raw_names"])),
            f"{prefix}_sources": " | ".join(sorted(payload["source_labels"])),
            f"{prefix}_source_files": " | ".join(sorted(payload["source_files"])),
            f"{prefix}_occurrence_count": payload["occurrence_count"],
            f"{prefix}_unique_file_count": len(payload["source_files"]),
        })

    if not records:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(records, columns=columns)

output_store = defaultdict(make_empty_entry)
truth_store = defaultdict(make_empty_entry)

output_files = sorted(OUTPUTS_ROOT.glob("**/*.json"))
truth_files = sorted(TRUTH_ROOT.glob("*/*_outcome.json"))
skipped_output_files = []
skipped_truth_files = []

for output_file in output_files:
    relative_parts = output_file.relative_to(OUTPUTS_ROOT).parts
    if not relative_parts:
        continue
    conference = relative_parts[0]
    data = load_json(output_file)
    if data is None:
        skipped_output_files.append(str(output_file))
        continue
    for _, raw_name, source_label, source_path in collect_output_name_rows(data, conference, output_file):
        register_occurrence(output_store, conference, raw_name, source_label, source_path)

for truth_file in truth_files:
    conference = truth_file.parent.name
    data = load_json(truth_file)
    if data is None:
        skipped_truth_files.append(str(truth_file))
        continue
    for _, raw_name, source_label in extract_truth_name_rows(data, conference):
        register_occurrence(truth_store, conference, raw_name, source_label, truth_file)

output_df = store_to_dataframe(output_store, "output")
truth_df = store_to_dataframe(truth_store, "truth")

output_df = output_df.sort_values(["conference", "normalized_name"]).reset_index(drop=True) if not output_df.empty else output_df
truth_df = truth_df.sort_values(["conference", "normalized_name"]).reset_index(drop=True) if not truth_df.empty else truth_df

if skipped_output_files:
    print(f"Skipped {len(skipped_output_files)} invalid output JSON files.")
if skipped_truth_files:
    print(f"Skipped {len(skipped_truth_files)} invalid truth JSON files.")

output_df.head(), truth_df.head()

Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2020NES/output_2020_11_06_NES_S6/6_CO2_Reduction_Zoom_Meeting_2020_11_06_08_45_55/ATTN_6_CO2_Reduction_Zoom_Meeting_2020_11_06_08_45_55.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021MND/output_2021_04_22_MND_S8/ATTN_Zoom_Meeting_Room_6_2021_04_22_13_00_55_chunk2.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021MND/output_2021_04_22_MND_S8/ATTN_Zoom_Meeting_Room_6_2021_04_22_13_18_39_chunk1.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021MND/output_2021_04_23_MND_S11/ATTN_bot2_Zoom_Meeting_2021_04_23_13_13_56_chunk4.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_a

(  conference  normalized_name output_raw_names  \
 0    2020NES  adam holewinski  Adam Holewinski   
 1    2020NES      alissa park      Alissa Park   
 2    2020NES     andrea hicks     Andrea Hicks   
 3    2020NES           andrew           Andrew   
 4    2020NES      andrew feig      Andrew Feig   
 
                                       output_sources  \
 0  chunk_summary.speaking_time_seconds | session_...   
 1  chunk_summary.speaking_time_seconds | session_...   
 2  chunk_summary.speaking_time_seconds | session_...   
 3  chunk_summary.speaking_time_seconds | session_...   
 4  chunk_summary.speaking_time_seconds | session_...   
 
                                  output_source_files  output_occurrence_count  \
 0  outputs/2020NES/output_2020_11_05_NES_S4/4_Bey...                       92   
 1  outputs/2020NES/output_2020_11_05_NES_S1/1_DAC...                      140   
 2  outputs/2020NES/output_2020_11_05_NES_S5/5_Dec...                       49   
 3  outputs/2020NES/

In [20]:
matched_df = output_df.merge(
    truth_df,
    on=["conference", "normalized_name"],
    how="inner",
    suffixes=("_output", "_truth"),
)

unmatched_output_df = output_df.merge(
    truth_df[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_output_df = unmatched_output_df[unmatched_output_df["_merge"] == "left_only"].drop(columns=["_merge"])

unmatched_truth_df = truth_df.merge(
    output_df[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_truth_df = unmatched_truth_df[unmatched_truth_df["_merge"] == "left_only"].drop(columns=["_merge"])

conference_order = sorted(set(output_df["conference"].unique()).union(truth_df["conference"].unique())) if not output_df.empty or not truth_df.empty else []
progress_rows = []

for conference in conference_order:
    output_conference = output_df[output_df["conference"] == conference] if not output_df.empty else pd.DataFrame()
    truth_conference = truth_df[truth_df["conference"] == conference] if not truth_df.empty else pd.DataFrame()
    matched_conference = matched_df[matched_df["conference"] == conference] if not matched_df.empty else pd.DataFrame()

    output_unique = len(output_conference)
    truth_unique = len(truth_conference)
    matched_unique = len(matched_conference)

    progress_rows.append({
        "conference": conference,
        "output_unique_participants": output_unique,
        "truth_unique_participants": truth_unique,
        "matched_participants": matched_unique,
        "unmatched_output_participants": output_unique - matched_unique,
        "unmatched_truth_participants": truth_unique - matched_unique,
        "output_match_rate": round(matched_unique / output_unique, 4) if output_unique else 0.0,
        "truth_match_rate": round(matched_unique / truth_unique, 4) if truth_unique else 0.0,
    })

progress_df = pd.DataFrame(progress_rows).sort_values("conference").reset_index(drop=True) if progress_rows else pd.DataFrame(
    columns=[
        "conference",
        "output_unique_participants",
        "truth_unique_participants",
        "matched_participants",
        "unmatched_output_participants",
        "unmatched_truth_participants",
        "output_match_rate",
        "truth_match_rate",
    ]
)

matched_output_path = BASE / "matched_output_participants.csv"
unmatched_output_path = BASE / "unmatched_output_participants.csv"
unmatched_truth_path = BASE / "unmatched_truth_participants.csv"
progress_path = BASE / "participant_matching_progress_by_conference.csv"

matched_df.to_csv(matched_output_path, index=False)
unmatched_output_df.to_csv(unmatched_output_path, index=False)
unmatched_truth_df.to_csv(unmatched_truth_path, index=False)
progress_df.to_csv(progress_path, index=False)

print(f"Saved: {matched_output_path}")
print(f"Saved: {unmatched_output_path}")
print(f"Saved: {unmatched_truth_path}")
print(f"Saved: {progress_path}")

print(f"Output participants: {len(output_df)}")
print(f"Truth participants: {len(truth_df)}")
print(f"Matched participants: {len(matched_df)}")

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/matched_output_participants.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_output_participants.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_truth_participants.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_matching_progress_by_conference.csv
Output participants: 772
Truth participants: 334
Matched participants: 291


In [ ]:
def build_unmatched_review(df, label):
    if df.empty:
        return pd.DataFrame(
            columns=[
                "normalized_name",
                "record_type",
                "conferences",
                "raw_names",
                "source_files",
                "occurrence_count",
                "unique_file_count",
            ]
        )

    raw_name_column = f"{label}_raw_names"
    source_file_column = f"{label}_source_files"
    occurrence_column = f"{label}_occurrence_count"

    review_df = (
        df.assign(
            raw_name=df[raw_name_column],
            source_file=df[source_file_column],
            occurrence=df[occurrence_column],
            record_type=label,
        )
        .groupby("normalized_name", as_index=False)
        .agg(
            record_type=("record_type", lambda values: " | ".join(sorted(set(values)))),
            conferences=("conference", lambda values: " | ".join(sorted(set(values)))),
            raw_names=("raw_name", lambda values: " | ".join(sorted(set(filter(None, values))))),
            source_files=("source_file", lambda values: " | ".join(sorted(set(filter(None, values))))),
            occurrence_count=("occurrence", "sum"),
        )
    )
    review_df["unique_file_count"] = review_df["source_files"].apply(lambda value: 0 if not value else len(value.split(" | ")))
    return review_df.sort_values(["conferences", "normalized_name"]).reset_index(drop=True)

unmatched_output_review_df = build_unmatched_review(unmatched_output_df, "output")
unmatched_truth_review_df = build_unmatched_review(unmatched_truth_df, "truth")

unmatched_review_df = (
    pd.concat([unmatched_output_review_df, unmatched_truth_review_df], ignore_index=True)
    .sort_values(["conferences", "normalized_name"])
    .reset_index(drop=True)
    if not unmatched_output_review_df.empty or not unmatched_truth_review_df.empty
    else pd.DataFrame(
        columns=[
            "normalized_name",
            "record_type",
            "conferences",
            "raw_names",
            "source_files",
            "occurrence_count",
            "unique_file_count",
        ]
    )
)

unmatched_review_path = BASE / "unmatched_name_review.csv"
unmatched_review_df.to_csv(unmatched_review_path, index=False)
print(f"Saved: {unmatched_review_path}")

def build_manual_alias_template(review_df):
    if review_df.empty:
        return pd.DataFrame(
            columns=[
                "alias_name",
                "canonical_name",
                "record_type",
                "conferences",
                "raw_names",
                "source_files",
                "notes",
            ]
        )

    template_df = review_df.copy()
    template_df = template_df.rename(columns={"normalized_name": "alias_name"})
    template_df["canonical_name"] = ""
    template_df["notes"] = ""
    template_df = template_df[[
        "alias_name",
        "canonical_name",
        "record_type",
        "conferences",
        "raw_names",
        "source_files",
        "notes",
    ]]
    return template_df

manual_alias_template_path = BASE / "participant_alias_mapping_template.csv"
if not manual_alias_template_path.exists():
    manual_alias_template_df = build_manual_alias_template(unmatched_review_df)
    manual_alias_template_df.to_csv(manual_alias_template_path, index=False)
    print(f"Saved: {manual_alias_template_path}")
else:
    print(f"Template already exists: {manual_alias_template_path}")

print("If you want the notebook to use manual aliases on the next run, create or edit:")
print(f"- {REPO_ROOT / 'analysis_v1' / 'participant_alias_mapping.csv'}")
print("Use the template as a starting point.")

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_name_review.csv


In [22]:
display_columns = [
    "conference",
    "normalized_name",
    "output_raw_names",
    "truth_raw_names",
]

print("Matched participants by conference:")
display(progress_df)

print("Unmatched name review sample:")
display(unmatched_review_df.head(20))

print("Matched participant sample:")
display(matched_df[display_columns].head(20) if not matched_df.empty else matched_df)

Matched participants by conference:


,conference,output_unique_participants,truth_unique_participants,matched_participants,unmatched_output_participants,unmatched_truth_participants,output_match_rate,truth_match_rate
0,2020NES,112,52,51,61,1,0.4554,0.9808
1,2021ABI,139,44,36,103,8,0.2590,0.8182
2,2021CMC,85,37,32,53,5,0.3765,0.8649
3,2021MND,108,40,38,70,2,0.3519,0.9500
4,2021MZT,82,43,30,52,13,0.3659,0.6977
5,2021NES,99,53,47,52,6,0.4747,0.8868
6,2021SLU,88,41,34,54,7,0.3864,0.8293
7,2022MND,59,24,23,36,1,0.3898,0.9583


Unmatched name review sample:


,normalized_name,record_type,conferences,raw_names,source_files,occurrence_count,unique_file_count
0,andrew,output,2020NES,Andrew,outputs/2020NES/output_2020_11_06_NES_S8/2020_...,12,8
1,angela hagen,output,2020NES,Angela Hagen,outputs/2020NES/output_2020_11_06_NES_S6/6_CO2...,3,1
2,ashleigh baber,output,2020NES,Ashleigh Baber,outputs/2020NES/output_2020_11_05_NES_S1/1_DAC...,37,14
3,betsy cantwell,output,2020NES,Betsy Cantwell,outputs/2020NES/output_2020_11_06_NES_S2/2020_...,11,9
4,cheng liu,output,2020NES,Cheng Liu,outputs/2020NES/output_2020_11_06_NES_S3/2020_...,6,4
5,christopher gorski,output,2020NES,Christopher Gorski,outputs/2020NES/output_2020_11_05_NES_S1/1_DAC...,34,14
6,daniel,output,2020NES,Daniel,outputs/2020NES/output_2020_11_06_NES_S8/2020_...,9,7
7,daniel yawitz,output,2020NES,Daniel Yawitz,outputs/2020NES/output_2020_11_05_NES_S4/4_Bey...,29,15
8,david gator kwab,output,2020NES,David Gator Kwab,outputs/2020NES/output_2020_11_06_NES_S12/6_ne...,9,3
9,fateme rezaei,output,2020NES,Fateme Rezaei,outputs/2020NES/output_2020_11_05_NES_S1/1_DAC...,42,15


Matched participant sample:


,conference,normalized_name,output_raw_names,truth_raw_names
0,2020NES,adam holewinski,Adam Holewinski,Adam Holewinski
1,2020NES,andrea hicks,Andrea Hicks,Andrea Hicks
2,2020NES,andrew teixeira,Andrew Teixeira,Andrew Teixeira
3,2020NES,betar gallant,Betar Gallant,Betar Gallant
4,2020NES,burcu gurkan,Burcu Gurkan,Burcu Gurkan
5,2020NES,caleb hill,Caleb Hill,Caleb Hill
6,2020NES,carlos morales guio,Carlos Morales Guio | Carlos Morales-Guio,Carlos Morales-Guio
7,2020NES,charles mccrory,Charles McCrory,Charles McCrory
8,2020NES,chong liu,Chong Liu,Chong Liu
9,2020NES,chris gorski,Chris Gorski,Chris Gorski
